# Text splitting for retrieval

Text splitting turns a long document into smaller passages that can be indexed and retrieved independently. A useful chunk should be small enough to search precisely, but large enough to keep the context needed to understand its answer.

This notebook builds a small character-based splitter from scratch so its behavior is easy to inspect. Production projects can use a language-aware splitter when boundaries such as paragraphs or sentences matter.

## Choose chunk size and overlap

- **Chunk size** is the maximum number of characters in a passage in this example.
- **Overlap** repeats a short suffix at the beginning of the next passage. This helps preserve a thought that crosses a boundary.
- Excessive overlap creates redundant passages; very small chunks can lose useful context.

Character counts are only a teaching approximation. Language models have token limits, so real systems often measure tokens and split on semantic boundaries.

In [ ]:
document = (
    "A library is adding a reservation service. Members can reserve a book online. "
    "A reservation remains active for two days after the book becomes available. "
    "If it is not collected in time, the next member in the queue is notified. "
    "Members can cancel a reservation from their account before collection."
)

chunk_size = 100
overlap = 20
print(f"Document length: {len(document)} characters")

In [ ]:
def split_into_chunks(text, size, overlap):
    """Split text into fixed-width slices with a repeated boundary region."""
    if size <= 0:
        raise ValueError("size must be positive")
    if overlap < 0 or overlap >= size:
        raise ValueError("overlap must be between zero and size - 1")

    step = size - overlap
    return [text[start : start + size] for start in range(0, len(text), step)]


chunks = split_into_chunks(document, chunk_size, overlap)
for index, chunk in enumerate(chunks, start=1):
    print(f"Chunk {index} ({len(chunk)} characters): {chunk!r}")

In [ ]:
# Neighboring chunks share the configured number of characters.
for left, right in zip(chunks, chunks[1:]):
    assert left[-overlap:] == right[:overlap]

assert "".join(chunks) != document  # Overlap means simple concatenation repeats text.
print(f"Created {len(chunks)} overlapping chunks successfully.")

## Improve boundary quality

The fixed-width example can cut a sentence in half. A practical splitter first looks for a paragraph break, then a line break or sentence boundary, and only falls back to character boundaries when needed. After choosing a strategy, inspect representative chunks and test whether each passage can answer a question without relying on unrelated text.

### Try it

1. Change `chunk_size` and compare the number of resulting passages.
2. Set `overlap` to `0` and inspect where context is lost.
3. Replace the sample with a short policy or FAQ and check whether complete ideas stay together.